This Jupyter notebook identifies BGC-associated co-expression modules by testing each BGC–module pair within each strain using Fisher’s exact test. For significant associations, it calculates BGC coverage, module purity, and the number of BGC genes assigned to the module. Modules are then classified as BGC-related only if they pass the combined thresholds for FDR, BGC coverage, module purity, and minimum BGC gene count, thereby filtering out broad modules dominated by global transcriptional processes.

In [213]:
import pandas as pd

In [214]:
import pandas as pd
from pathlib import Path

base_dir = Path("/Users/annasve/Desktop/article_data/expression_module")

summary_rows = []
total_old_bgcs = set()
total_new_bgcs = set()

all_dfs = []

# keywords of columns to exclude (matches e.g. 216-DNPM-1, 216-ISP2-2, etc.)
EXCLUDE_KEYWORDS = ["DNPM", "ISP2", "MA", "SoyM", "TSB", "gluc", "gly", "malt"]

for strain_dir in base_dir.iterdir():
    if not strain_dir.is_dir():
        continue

    strain = strain_dir.name
    old_bgcs = set()
    new_bgcs = set()

    for file in strain_dir.glob("*borders.xlsx"):
        df = pd.read_excel(file)

        # ---- BGC counting ----
        if "Region_Nr" in df.columns:
            old_bgcs.update(df["Region_Nr"].dropna().astype(str))

        if "refined_BGC_ID" in df.columns:
            new_bgcs.update(df["refined_BGC_ID"].dropna().astype(str))

        # ---- add Strain + Strain_Module ----
        df["Strain"] = strain
        if "Module" in df.columns:
            df["Strain_Module"] = strain + "_" + df["Module"].astype(str)

        # ---- drop expression columns ----
        drop_cols = [
            c for c in df.columns
            if any(k.lower() in c.lower() for k in EXCLUDE_KEYWORDS)
        ]
        df_small = df.drop(columns=drop_cols)

        all_dfs.append(df_small)

    summary_rows.append({
        "Strain": strain,
        "Original_BGCs": len(old_bgcs),
        "Refined_BGCs": len(new_bgcs)
    })

    total_old_bgcs.update(old_bgcs)
    total_new_bgcs.update(new_bgcs)

summary_df = pd.DataFrame(summary_rows).sort_values("Strain")

# Concatenated dataframe (all strains/files)
all_df = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()

print("Per-strain summary:")
print(summary_df)

print("\nOverall totals:")
print(f"Total original BGCs: {len(total_old_bgcs)}")
print(f"Total refined BGCs: {len(total_new_bgcs)}")
print(f"Net change: {len(total_new_bgcs) - len(total_old_bgcs)}")

print("\nConcatenated dataframe shape (after dropping expression cols):", all_df.shape)

# Optional: save outputs
#summary_df.to_csv(base_dir / "bgc_refinement_summary_per_strain.csv", index=False)
#all_df.to_parquet(base_dir / "all_strains_borders_no_expression_cols.parquet", index=False)


Per-strain summary:
        Strain  Original_BGCs  Refined_BGCs
123  NBC_00001             28            33
60   NBC_00003             39            55
78   NBC_00012             27            34
80   NBC_00024             32            35
3    NBC_00028             27            31
..         ...            ...           ...
52   NBC_01788             23            27
72   NBC_01790             24            33
111  NBC_01803             28            35
30   NBC_01814             29            42
18   NBC_01815             23            25

[133 rows x 3 columns]

Overall totals:
Total original BGCs: 50
Total refined BGCs: 59
Net change: 9

Concatenated dataframe shape (after dropping expression cols): (1062528, 60)


In [215]:
total_original = summary_df["Original_BGCs"].sum()
total_refined = summary_df["Refined_BGCs"].sum()

print(f"Total Original BGCs: {total_original}")
print(f"Total Refined BGCs: {total_refined}")
print(f"Net change: {total_refined - total_original}")


Total Original BGCs: 4012
Total Refined BGCs: 4826
Net change: 814


In [216]:
all_df.head()

,Geneid,Product,Sequence_Length,Gene_Kind,Is_Core_Gene,Region_Nr,corr_to_bgc,expressed_any,high_corr,refined_member,...,Cluster_Product_6,Cand_Cluster_7,Cluster_Product_7,Cand_Cluster_8,Cluster_Product_8,Cand_Cluster_9,Cluster_Product_9,Cand_Cluster_10,Cluster_Product_10,Unnamed: 0
0,OHB55_00005,NaN,NaN,NaN,False,NaN,NaN,True,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,OHB55_00005_2,NaN,NaN,NaN,False,NaN,NaN,True,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,OHB55_00010,hypothetical protein,100.0,NaN,False,NaN,NaN,True,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,OHB55_00015,efflux RND transporter periplasmic adaptor sub...,460.0,NaN,False,NaN,NaN,True,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,OHB55_00020,ABC transporter ATP-binding protein,319.0,NaN,False,NaN,NaN,True,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [217]:
all_df.columns

Index(['Geneid', 'Product', 'Sequence_Length', 'Gene_Kind', 'Is_Core_Gene',
       'Region_Nr', 'corr_to_bgc', 'expressed_any', 'high_corr',
       'refined_member', 'refined_BGC_ID', 'Module', 'kME', 'Cand_Cluster_1',
       'Cluster_Product_1', 'Cand_Cluster_2', 'Cluster_Product_2',
       'Cand_Cluster_3', 'Cluster_Product_3', 'Cand_Cluster_4',
       'Cluster_Product_4', 'prediction', 'score', 'Description',
       'Preferred_name', 'COG_category', 'PFAMs', 'KEGG_Module', 'Orthogroup',
       'CAI', 'CBI', 'Strain', 'Group_ID', 'Locus_ID', 'Overlap_Lengths',
       'Strain_Module', 'motif_id', 'motif_name', 'start', 'site_sequence',
       'E_value', 'significant_site', 'BGC_region', 'BGC_block', '__gorder__',
       'low_expression_rule', 'Region_Nr_original', 'Cand_Cluster_5',
       'Cluster_Product_5', 'Cand_Cluster_6', 'Cluster_Product_6',
       'Cand_Cluster_7', 'Cluster_Product_7', 'Cand_Cluster_8',
       'Cluster_Product_8', 'Cand_Cluster_9', 'Cluster_Product_9',
       '

In [218]:
all_df["Strain"] = all_df["Strain_Module"].str.split("_").str[:2].str.join("_")

In [219]:
all_df.head()

,Geneid,Product,Sequence_Length,Gene_Kind,Is_Core_Gene,Region_Nr,corr_to_bgc,expressed_any,high_corr,refined_member,...,Cluster_Product_6,Cand_Cluster_7,Cluster_Product_7,Cand_Cluster_8,Cluster_Product_8,Cand_Cluster_9,Cluster_Product_9,Cand_Cluster_10,Cluster_Product_10,Unnamed: 0
0,OHB55_00005,NaN,NaN,NaN,False,NaN,NaN,True,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,OHB55_00005_2,NaN,NaN,NaN,False,NaN,NaN,True,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,OHB55_00010,hypothetical protein,100.0,NaN,False,NaN,NaN,True,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,OHB55_00015,efflux RND transporter periplasmic adaptor sub...,460.0,NaN,False,NaN,NaN,True,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,OHB55_00020,ABC transporter ATP-binding protein,319.0,NaN,False,NaN,NaN,True,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Universal BGC IDs

In [220]:
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

In [221]:
cluster_product_cols = [c for c in all_df.columns if c.startswith("Cluster_Product")]

for c in cluster_product_cols:
    # keep NA as NA; only operate on real strings
    m = all_df[c].notna()
    all_df.loc[m, c] = (
        all_df.loc[m, c]
        .astype(str)
        .str.replace(", ", "_", regex=False)
        .str.strip()
    )

# also strip Module/Strain to avoid hidden whitespace
for c in ["Strain", "Module"]:
    if c in all_df.columns:
        m = all_df[c].notna()
        all_df.loc[m, c] = all_df.loc[m, c].astype(str).str.strip()


In [222]:
all_df["BGC_ID_old"] = pd.NA

mask_old = (
    all_df["Strain"].notna() &
    all_df["Region_Nr"].notna() &
    all_df["Cluster_Product_1"].notna()
)

# IMPORTANT: build only on the masked rows to avoid "nan" strings
all_df.loc[mask_old, "BGC_ID_old"] = (
    all_df.loc[mask_old, "Strain"].astype(str) + "_" +
    all_df.loc[mask_old, "Region_Nr"].astype(str) + "_" +
    all_df.loc[mask_old, "Cluster_Product_1"].astype(str)
)


In [223]:
def dominant_cluster_product(df):
    prod_df = df[cluster_product_cols].dropna(how="all")
    if prod_df.empty:
        return pd.NA

    # flatten and keep only real values
    flat = pd.Series(prod_df.values.ravel()).dropna()

    # remove empty strings and any accidental 'nan' string
    flat = flat[flat.astype(str).str.strip().ne("")]
    flat = flat[flat.astype(str).str.lower().ne("nan")]

    if flat.empty:
        return pd.NA

    return flat.value_counts().idxmax()


In [224]:
bgc_new_map = (
    all_df
    .dropna(subset=["Strain", "refined_BGC_ID"])
    .groupby(["Strain", "refined_BGC_ID"], dropna=True)
    .apply(dominant_cluster_product)
    .reset_index(name="Dominant_Product")
)

bgc_new_map["BGC_ID_new"] = pd.NA
mask_new = (
    bgc_new_map["Strain"].notna() &
    bgc_new_map["refined_BGC_ID"].notna() &
    bgc_new_map["Dominant_Product"].notna()
)

bgc_new_map.loc[mask_new, "BGC_ID_new"] = (
    bgc_new_map.loc[mask_new, "Strain"].astype(str) + "_" +
    bgc_new_map.loc[mask_new, "refined_BGC_ID"].astype(str) + "_" +
    bgc_new_map.loc[mask_new, "Dominant_Product"].astype(str)
)

all_df = all_df.merge(
    bgc_new_map[["Strain", "refined_BGC_ID", "BGC_ID_new"]],
    on=["Strain", "refined_BGC_ID"],
    how="left"
)


In [225]:
bad_tokens = ["_nan_", "_NA_", "_<NA>_", "nan", "<NA>", "NA"]

def is_bad_id(s):
    if pd.isna(s):
        return False
    ss = str(s)
    return any(t in ss for t in bad_tokens)

for col in ["BGC_ID_old", "BGC_ID_new"]:
    m = all_df[col].apply(is_bad_id)
    all_df.loc[m, col] = pd.NA


### Fishers test

In [226]:
from scipy.stats import fisher_exact
import pandas as pd

In [227]:
def fisher_bgc_enrichment(
    df_strain,
    bgc_col,
    bgc_id,
    module
):
    """
    df_strain: all_df filtered to ONE strain
    bgc_col: 'Region_Nr' or 'refined_BGC_ID'
    bgc_id: specific BGC ID
    module: module name
    """

    in_bgc = df_strain[bgc_col] == bgc_id
    in_module = df_strain["Module"] == module

    a = (in_bgc & in_module).sum()
    b = (in_bgc & ~in_module).sum()
    c = (~in_bgc & in_module).sum()
    d = (~in_bgc & ~in_module).sum()

    if a == 0:
        return None  # no enrichment possible

    odds_ratio, pval = fisher_exact([[a, b], [c, d]], alternative="greater")

    return {
        "BGC_ID": bgc_id,
        "Module": module,
        "a_in_bgc_in_module": a,
        "bgc_size": a + b,
        "module_size": a + c,
        "strain_genes": a + b + c + d,
        "odds_ratio": odds_ratio,
        "p_value": pval
    }


### Fishers old BGCs

In [228]:
results = []

for strain, df_strain in all_df.groupby("Strain"):
    modules = df_strain["Module"].dropna().unique()
    bgcs = df_strain["BGC_ID_old"].dropna().unique()

    for bgc in bgcs:
        for module in modules:
            res = fisher_bgc_enrichment(
                df_strain,
                bgc_col="BGC_ID_old",
                bgc_id=bgc,
                module=module
            )
            if res:
                res["Strain"] = strain
                res["BGC_type"] = "original"
                results.append(res)


### Fishers new BGCs

In [229]:
for strain, df_strain in all_df.groupby("Strain"):
    modules = df_strain["Module"].dropna().unique()
    bgcs = df_strain["BGC_ID_new"].dropna().unique()

    for bgc in bgcs:
        for module in modules:
            res = fisher_bgc_enrichment(
                df_strain,
                bgc_col="BGC_ID_new",
                bgc_id=bgc,
                module=module
            )
            if res:
                res["Strain"] = strain
                res["BGC_type"] = "refined"
                results.append(res)


In [230]:
fishers_df = pd.DataFrame(results)

from statsmodels.stats.multitest import multipletests

fishers_df["FDR"] = multipletests(
    fishers_df["p_value"],
    method="fdr_bh"
)[1]


In [231]:
fishers_df.head()

,BGC_ID,Module,a_in_bgc_in_module,bgc_size,module_size,strain_genes,odds_ratio,p_value,Strain,BGC_type,FDR
0,NBC_00001_1.0_T3PKS,P1_I26_M11,2,45,1196,9830,0.334658,0.979116,NBC_00001,original,0.989948
1,NBC_00001_1.0_T3PKS,P1_I26_M44,6,45,177,9830,8.649573,0.000142,NBC_00001,original,0.002051
2,NBC_00001_1.0_T3PKS,P1_I26_M13,2,45,695,9830,0.610222,0.837429,NBC_00001,original,0.881420
3,NBC_00001_1.0_T3PKS,P1_I26_M10,7,45,539,9830,3.203947,0.010589,NBC_00001,original,0.066721
4,NBC_00001_1.0_T3PKS,UNCLASSIFIED,4,45,331,9830,2.821810,0.063879,NBC_00001,original,0.191264


In [232]:
sig_df = fishers_df[fishers_df["FDR"] < 0.001].copy()

In [233]:
sig_df.head(50)

,BGC_ID,Module,a_in_bgc_in_module,bgc_size,module_size,strain_genes,odds_ratio,p_value,Strain,BGC_type,FDR
40,NBC_00001_2.0_NRP-metallophore_NRPS,P1_I26_M17,19,44,195,9830,41.497727,1.691689e-21,NBC_00001,original,1.909188e-19
46,NBC_00001_3.0_terpene,P1_I26_M13,10,25,695,9830,8.875912,3.607295e-06,NBC_00001,original,7.772280e-05
75,NBC_00001_5.0_terpene,P1_I26_M30,7,23,193,9830,22.630040,1.895270e-07,NBC_00001,original,5.183215e-06
169,NBC_00001_11.0_furan,P1_I26_M15,8,23,221,9830,24.022535,2.109141e-08,NBC_00001,original,6.737113e-07
201,NBC_00001_14.0_terpene,P1_I26_M77,3,21,28,9830,65.226667,2.660082e-05,NBC_00001,original,4.676016e-04
212,NBC_00001_15.0_furan,P1_I26_M69,6,22,51,9830,81.358333,1.008445e-09,NBC_00001,original,3.869422e-08
262,NBC_00001_17.0_thioamitides,P2_I9_M21,6,22,49,9830,85.159884,7.852241e-10,NBC_00001,original,3.056745e-08
272,NBC_00001_18.0_NI-siderophore,P1_I26_M38,4,9,43,9830,200.656410,3.936207e-08,NBC_00001,original,1.201987e-06
323,NBC_00001_24.0_NRPS_T3PKS,P1_I26_M68,5,50,83,9830,13.820513,5.976345e-05,NBC_00001,original,9.610501e-04
325,NBC_00001_24.0_NRPS_T3PKS,P1_I26_M17,11,50,195,9830,14.709588,2.685429e-09,NBC_00001,original,9.787841e-08


In [234]:
mask = sig_df["BGC_ID"].notna()
sig_df.loc[mask, "BGC_ID"] = (
    sig_df.loc[mask, "BGC_ID"]
    .astype(str)
    .str.replace(".0", "", regex=False)
)


In [235]:
sig_refined = sig_df[sig_df["BGC_type"] == "refined"].copy()
sig_original = sig_df[sig_df["BGC_type"] == "original"].copy()

In [238]:
sig_original.to_excel('/Users/annasve/Desktop/article_data/output/iterative_WGCNA/analysis/fishers_001_original.xlsx', index = False)

In [239]:
sig_refined.to_excel('/Users/annasve/Desktop/article_data/output/iterative_WGCNA/analysis/fishers_001_refined.xlsx', index = False)

In [263]:
df = sig_df.copy()
df["bgc_coverage"] = df["a_in_bgc_in_module"] / df["bgc_size"]
df["module_purity"] = df["a_in_bgc_in_module"] / df["module_size"]

bgc_specific = df[
    (df["FDR"] < 0.01) &
    (df["bgc_coverage"] >= 0.4) &
    (df["module_purity"] >= 0.05)  # tune: 0.03–0.10
].copy()


In [264]:
bgc_specific

,BGC_ID,Module,a_in_bgc_in_module,bgc_size,module_size,strain_genes,odds_ratio,p_value,Strain,BGC_type,FDR,bgc_coverage,module_purity
40,NBC_00001_2_NRP-metallophore_NRPS,P1_I26_M17,19,44,195,9830,41.497727,1.691689e-21,NBC_00001,original,1.909188e-19,0.431818,0.097436
272,NBC_00001_18_NI-siderophore,P1_I26_M38,4,9,43,9830,200.656410,3.936207e-08,NBC_00001,original,1.201987e-06,0.444444,0.093023
1033,NBC_00003_36_NI-siderophore,P1_I23_M20,10,11,172,7852,474.012346,2.115220e-16,NBC_00003,original,1.609221e-14,0.909091,0.058140
1089,NBC_00003_39_ectoine,P1_I23_M27,8,9,58,7852,1246.880000,4.803727e-17,NBC_00003,original,3.858273e-15,0.888889,0.137931
1249,NBC_00012_15_ectoine,P3_I7_M31,4,10,33,8961,205.103448,3.151180e-08,NBC_00012,original,9.778054e-07,0.400000,0.121212
...,...,...,...,...,...,...,...,...,...,...,...,...,...
74863,NBC_01815_10_NRPS-like_butyrolactone_T1PKS_NRPS,P10_I1_M1,15,28,96,7062,99.045584,1.041987e-21,NBC_01815,refined,1.184877e-19,0.535714,0.156250
74891,NBC_01815_17_T1PKS,P10_I1_M2,15,16,47,7062,3287.812500,2.932979e-33,NBC_01815,refined,7.992314e-31,0.937500,0.319149
74895,NBC_01815_18_terpene_NRPS_ladderane,P4_I5_M3,5,11,33,7062,209.017857,7.354155e-10,NBC_01815,refined,2.868810e-08,0.454545,0.151515
74918,NBC_01815_21_transAT-PKS_NRPS-like_NRPS_PKS-like,P1_I34_M2,29,45,296,7062,45.821629,1.019616e-29,NBC_01815,refined,2.010709e-27,0.644444,0.097973


In [265]:
bgc_refined = bgc_specific[bgc_specific["BGC_type"] == "refined"].copy()
bgc_original = bgc_specific[bgc_specific["BGC_type"] == "original"].copy()

In [266]:
bgc_original

,BGC_ID,Module,a_in_bgc_in_module,bgc_size,module_size,strain_genes,odds_ratio,p_value,Strain,BGC_type,FDR,bgc_coverage,module_purity
40,NBC_00001_2_NRP-metallophore_NRPS,P1_I26_M17,19,44,195,9830,41.497727,1.691689e-21,NBC_00001,original,1.909188e-19,0.431818,0.097436
272,NBC_00001_18_NI-siderophore,P1_I26_M38,4,9,43,9830,200.656410,3.936207e-08,NBC_00001,original,1.201987e-06,0.444444,0.093023
1033,NBC_00003_36_NI-siderophore,P1_I23_M20,10,11,172,7852,474.012346,2.115220e-16,NBC_00003,original,1.609221e-14,0.909091,0.058140
1089,NBC_00003_39_ectoine,P1_I23_M27,8,9,58,7852,1246.880000,4.803727e-17,NBC_00003,original,3.858273e-15,0.888889,0.137931
1249,NBC_00012_15_ectoine,P3_I7_M31,4,10,33,8961,205.103448,3.151180e-08,NBC_00012,original,9.778054e-07,0.400000,0.121212
...,...,...,...,...,...,...,...,...,...,...,...,...,...
52459,NBC_01790_24_T1PKS_butyrolactone_other,P1_I13_M5,42,59,132,5945,159.105882,5.723101e-59,NBC_01790,original,7.524071e-56,0.711864,0.318182
52527,NBC_01803_4_T1PKS,P1_I13_M59,18,31,62,5283,163.888112,2.205555e-28,NBC_01803,original,4.041019e-26,0.580645,0.290323
52919,NBC_01814_8_furan,P1_I29_M20,18,19,82,8115,2259.000000,3.091004e-36,NBC_01814,original,1.024914e-33,0.947368,0.219512
53503,NBC_01815_17_T1PKS,P10_I1_M2,16,39,47,7062,156.903226,2.865206e-26,NBC_01815,original,4.568297e-24,0.410256,0.340426


In [267]:
bgc_refined.head(50)

,BGC_ID,Module,a_in_bgc_in_module,bgc_size,module_size,strain_genes,odds_ratio,p_value,Strain,BGC_type,FDR,bgc_coverage,module_purity
53620,NBC_00001_2_NRP-metallophore_NRPS,P1_I26_M17,18,25,195,9830,1.398741e+02,4.359695e-26,NBC_00001,refined,6.863498e-24,0.720000,0.092308
53703,NBC_00001_22_thioamitides,P2_I9_M21,6,9,49,9830,4.547907e+02,9.282862e-13,NBC_00001,refined,4.975893e-11,0.666667,0.122449
53707,NBC_00001_23_NI-siderophore,P1_I26_M38,5,7,43,9830,6.437500e+02,2.628547e-11,NBC_00001,refined,1.210666e-09,0.714286,0.116279
53712,NBC_00001_26_ectoine,P1_I26_M78,2,3,26,9830,8.169167e+02,2.014949e-05,NBC_00001,refined,3.651615e-04,0.666667,0.076923
53749,NBC_00003_1_NRPS-like,P1_I23_M59,5,7,24,7852,1.029737e+03,3.578720e-12,NBC_00003,refined,1.812017e-10,0.714286,0.208333
53805,NBC_00003_10_NRPS-like_T3PKS,P1_I23_M44,16,26,44,7852,4.456000e+02,2.177782e-31,NBC_00003,refined,5.021430e-29,0.615385,0.363636
53870,NBC_00003_22_thioamide-NRP_NRPS_ladderane,P1_I23_M52,3,6,55,7852,1.498846e+02,6.409129e-06,NBC_00003,refined,1.304402e-04,0.500000,0.054545
53928,NBC_00003_33_melanin,P1_I23_M23,5,11,76,7852,9.119718e+01,3.283515e-08,NBC_00003,refined,1.015924e-06,0.454545,0.065789
53940,NBC_00003_37_lanthipeptide-class-iii_arylpolyene,P1_I23_M92,3,4,12,7852,2.613000e+03,1.090148e-08,NBC_00003,refined,3.643729e-07,0.750000,0.250000
53963,NBC_00003_41_lanthipeptide-class-iii,P1_I23_M23,4,5,76,7852,4.319444e+02,4.023564e-08,NBC_00003,refined,1.225168e-06,0.800000,0.052632


In [268]:
bgc_original.to_excel('/Users/annasve/Desktop/article_data/output/iterative_WGCNA/analysis/fishers_001_original_purity.xlsx', index = False)

In [269]:
bgc_refined.to_excel('/Users/annasve/Desktop/article_data/output/iterative_WGCNA/analysis/fishers_001_refined_purity.xlsx', index = False)